# Grounded RAG with Verifiable Citations (Grok)

Retrieval-augmented generation (RAG) is supposed to keep a model honest by feeding it real sources. But
"I gave it the documents" is not the same as "the answer is actually supported by them." Two failures
slip through constantly:

- **Uncited claims** — the answer states a fact that appears in *none* of the retrieved chunks (the model
  filled a gap from memory).
- **Mis-cited claims** — the answer cites `[doc_3]`, but `doc_3` doesn't actually contain that fact.

You can catch both with a **deterministic, zero-token verifier** that runs *after* Grok answers and
*before* you show the answer to a user. Every sentence must cite a chunk, and the cited chunk must
actually contain support for it. If it doesn't, we **fail closed** — regenerate or refuse — instead of
shipping a confident hallucination.

This cookbook builds a small end-to-end grounded-RAG loop on Grok:

1. **Retrieve** the most relevant chunks with a dependency-free lexical retriever.
2. **Generate** an answer that must cite its sources inline as `[chunk_id]`.
3. **Verify** — deterministically — that every claim is cited *and* each citation is genuinely
   supported by the chunk it points to.
4. **Repair or refuse** when verification fails.

Everything but the model call is pure Python: no token cost, fully reproducible. It builds directly on
the *grounding check* idea from
[Deterministic Guardrails for Grok Tool Calls](../deterministic_tool_call_guardrails/guide.ipynb),
turning it into a full retrieval recipe.


## Setup

Same OpenAI-compatible client the rest of the cookbook uses, pointed at the xAI endpoint. The retriever and verifier are pure Python, so this notebook runs end-to-end even without a key — in that case the model call falls back to a clearly-labeled offline demo response.

In [ ]:
%pip install openai python-dotenv --quiet

In [2]:
import os, json, re
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
XAI_API_KEY = os.environ.get("XAI_API_KEY", "")
MODEL = "grok-4"
LIVE = bool(XAI_API_KEY)
client = OpenAI(base_url="https://api.x.ai/v1", api_key=XAI_API_KEY) if LIVE else None
print("Mode:", "LIVE (real Grok API)" if LIVE else "OFFLINE DEMO (deterministic retriever + verifier run for real)")

Mode: OFFLINE DEMO (deterministic retriever + verifier run for real)


## A tiny corpus

For a self-contained example we use a handful of short chunks. In production these come from your
vector store or search index — the retriever and verifier below don't care where the chunks originate,
only that each has a stable `id` and some `text`.

In [3]:
CORPUS = [
    {"id": "grok_ctx", "text": "Grok 4 supports a context window of 256000 tokens."},
    {"id": "grok_tools", "text": "Grok exposes function calling via an OpenAI-compatible API at https://api.x.ai/v1."},
    {"id": "grok_stream", "text": "Responses from the xAI API can be streamed incrementally by setting stream=True."},
    {"id": "grok_struct", "text": "Structured outputs let you constrain Grok to return JSON that matches a schema."},
    {"id": "unrelated", "text": "The Eiffel Tower is located in Paris and was completed in 1889."},
]
CHUNKS = {c["id"]: c["text"] for c in CORPUS}
print(f"{len(CORPUS)} chunks loaded")

5 chunks loaded


## Step 1 — Retrieve (dependency-free)

A production system would use embeddings; to keep this notebook to a single `pip install` we use a
simple lexical overlap score. Swap this function for your vector search — nothing downstream changes.

In [4]:
def tokenize(s: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", s.lower()))


def retrieve(query: str, k: int = 3) -> list[dict]:
    # Score on content words only (drop short/function words) so an off-topic query retrieves nothing.
    q = {w for w in tokenize(query) if len(w) > 3}
    scored = [(len(q & tokenize(c["text"])), c) for c in CORPUS]
    scored = [(s, c) for s, c in scored if s > 0]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in scored[:k]]


retrieved = retrieve("How big is Grok's context window and how do I call tools?")
for c in retrieved:
    print(f"  [{c['id']}] {c['text']}")

  [grok_ctx] Grok 4 supports a context window of 256000 tokens.
  [grok_tools] Grok exposes function calling via an OpenAI-compatible API at https://api.x.ai/v1.
  [grok_struct] Structured outputs let you constrain Grok to return JSON that matches a schema.


## Step 2 — Generate with mandatory inline citations

We instruct Grok to answer **only** from the retrieved chunks and to tag every sentence with the
`[chunk_id]` it came from. The chunk ids are part of the prompt, so the model has a closed vocabulary of
citations to choose from.

In [5]:
SYSTEM = (
    "You answer strictly from the provided sources. Every sentence MUST end with a citation of the form "
    "[chunk_id] naming the source it came from. Use ONLY the chunk ids provided. If the sources do not "
    "contain the answer, reply exactly: INSUFFICIENT EVIDENCE."
)


def build_context(chunks: list[dict]) -> str:
    return "\n".join(f"[{c['id']}] {c['text']}" for c in chunks)


def grok_answer(query: str, chunks: list[dict]) -> str:
    context = build_context(chunks)
    user = f"Sources:\n{context}\n\nQuestion: {query}\nAnswer with inline [chunk_id] citations."
    if LIVE:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
            temperature=0,
        )
        return resp.choices[0].message.content
    # OFFLINE DEMO: a well-formed, cited answer grounded in the retrieved chunks.
    return ("Grok 4 supports a context window of 256000 tokens [grok_ctx]. "
            "You call tools through an OpenAI-compatible API at https://api.x.ai/v1 [grok_tools].")


query = "How big is Grok's context window and how do I call tools?"
answer = grok_answer(query, retrieved)
print(answer)

Grok 4 supports a context window of 256000 tokens [grok_ctx]. You call tools through an OpenAI-compatible API at https://api.x.ai/v1 [grok_tools].


## Step 3 — Verify citations deterministically (zero tokens)

This is the heart of the recipe. For every sentence in the answer we check:

1. **It has a citation** — an uncited factual sentence is rejected.
2. **The cited id exists** — no citing a chunk that wasn't retrieved.
3. **The citation is supported** — the sentence's salient content words must actually appear in the
   cited chunk. This is what turns a *citation* into a *verifiable* citation: it catches the model
   tagging a real chunk id onto a claim that chunk doesn't support.

The check is lexical and intentionally simple — it's a fast, free tripwire, not a proof. Tighten it
(exact-span quoting, entailment models) as your risk profile demands.

In [6]:
STOPWORDS = set("a an the of to and or is are was were be been being in on at for with by from as "
                "how do i you it this that these those can does support supports via using use".split())


def salient(sentence: str) -> set[str]:
    return {w for w in tokenize(sentence) if w not in STOPWORDS and len(w) > 2}


def verify_answer(answer: str, chunks: dict[str, str]) -> list[str]:
    """Return a list of problems. Empty list == the answer is fully grounded."""
    if answer.strip() == "INSUFFICIENT EVIDENCE":
        return []
    problems = []
    # split into sentences, keeping each sentence's trailing citations
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", answer) if s.strip()]
    for sent in sentences:
        cited = re.findall(r"\[([a-z0-9_]+)\]", sent)
        claim = re.sub(r"\[[a-z0-9_]+\]", "", sent)
        if not cited:
            problems.append(f"UNCITED: {claim.strip()!r}")
            continue
        terms = salient(claim)
        for cid in cited:
            if cid not in chunks:
                problems.append(f"BAD_ID: [{cid}] is not a retrieved chunk")
                continue
            support = tokenize(chunks[cid])
            # every citation must cover the majority of the claim's salient terms
            covered = terms & support
            if terms and len(covered) / len(terms) < 0.5:
                problems.append(f"UNSUPPORTED: {claim.strip()!r} -> [{cid}] "
                                f"(covers {len(covered)}/{len(terms)} key terms)")
    return problems


print("Verifying the model's answer...")
for p in verify_answer(answer, CHUNKS) or ["OK - fully grounded"]:
    print("  ", p)

Verifying the model's answer...
   OK - fully grounded


### The verifier earns its keep

Watch it reject three answers a naive pipeline would happily return: an uncited claim, a fabricated citation id, and a claim tagged onto a real-but-unrelated chunk.

In [7]:
bad_answers = {
    "uncited fact":       "Grok 4 supports a context window of 256000 tokens.",
    "hallucinated id":    "Grok runs entirely on-device [local_only].",
    "mis-cited claim":    "The Eiffel Tower is 330 meters tall [grok_ctx].",
}
for label, a in bad_answers.items():
    print(f"{label}:")
    for p in verify_answer(a, CHUNKS):
        print("   BLOCKED ->", p)

uncited fact:
   BLOCKED -> UNCITED: 'Grok 4 supports a context window of 256000 tokens.'
hallucinated id:
   BLOCKED -> BAD_ID: [local_only] is not a retrieved chunk
mis-cited claim:
   BLOCKED -> UNSUPPORTED: 'The Eiffel Tower is 330 meters tall .' -> [grok_ctx] (covers 0/5 key terms)


## Step 4 — The full grounded-RAG loop (retrieve → generate → verify → repair/refuse)

We tie it together with a fail-closed policy: if the answer doesn't verify, we make one bounded repair
attempt (re-ask, feeding the verifier's complaints back to the model), and if it still fails we refuse
rather than show an ungrounded answer.

In [8]:
def grounded_rag(query: str, max_repairs: int = 1) -> str:
    chunks = retrieve(query)
    if not chunks:
        return "INSUFFICIENT EVIDENCE (nothing retrieved)"
    answer = grok_answer(query, chunks)
    for attempt in range(max_repairs + 1):
        problems = verify_answer(answer, {c["id"]: c["text"] for c in chunks})
        if not problems:
            return answer
        if attempt == max_repairs:
            # Fail closed: never return an answer we could not verify.
            return "REFUSED: answer failed citation verification -> " + " | ".join(problems)
        # Bounded repair: hand the complaints back and regenerate (LIVE mode only; offline is fixed).
        if LIVE:
            fix = (f"Your previous answer had grounding problems: {problems}. "
                   f"Rewrite it, citing only supported chunks, or say INSUFFICIENT EVIDENCE.")
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": SYSTEM},
                          {"role": "user", "content": f"Sources:\n{build_context(chunks)}\n\n{fix}"}],
                temperature=0)
            answer = resp.choices[0].message.content
        else:
            break
    return answer


print("Q1:", grounded_rag("How big is Grok's context window and how do I call tools?"))
print()
print("Q2:", grounded_rag("What is the airspeed velocity of an unladen swallow?"))  # not in corpus -> refuse/insufficient

Q1: Grok 4 supports a context window of 256000 tokens [grok_ctx]. You call tools through an OpenAI-compatible API at https://api.x.ai/v1 [grok_tools].

Q2: INSUFFICIENT EVIDENCE (nothing retrieved)


## Recap

A grounded-RAG pipeline is only as trustworthy as its weakest citation. Three deterministic pieces make
Grok's answers auditable at zero token cost:

1. **Retrieve** relevant chunks (swap in your own vector search).
2. **Generate** with a closed vocabulary of `[chunk_id]` citations.
3. **Verify** that every claim is cited *and* genuinely supported by its chunk — then **fail closed**,
   repairing once or refusing rather than shipping an ungrounded answer.

Because the verifier is plain Python, it costs nothing to run on every response, is fully reproducible,
and gives you an audit trail of exactly which source backs each sentence. Harden it with span-level
quoting or an entailment model where the stakes are high.

**Next steps:** set `XAI_API_KEY` (see `.env.example`) and re-run — the same retriever and verifier now
wrap live Grok answers, and the repair loop re-asks the model when a citation doesn't hold up.